# Ablation MobileNetV3: GAP vs Bilinear vs HBP
Notebook ini menjalankan tiga seed pada split bersih yang sama.

In [ ]:
REPO_URL = "https://github.com/ediprin/coffee-bean-classification.git"
!git clone {REPO_URL} /content/coffee-bean-classification
%cd /content/coffee-bean-classification
!pip install -q -e .

In [ ]:
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU melalui Runtime > Change runtime type'
print(torch.cuda.get_device_name(0))
!python -m bilinear_lmmd.data.preparation.prepare_coffee17 --output data/coffee

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULT_ROOT = '/content/drive/MyDrive/coffee-bean-classification-hbp-ablation'

In [ ]:
from pathlib import Path
import subprocess

experiments = {
    'M0': 'configs/coffee17/M0_mobilenetv3_gap_source.yaml',
    'M0b': 'configs/coffee17/M0b_mobilenetv3_bilinear_source.yaml',
    'M1': 'configs/coffee17/M1_mobilenetv3_hbp_source.yaml',
}
seeds = [42, 123, 2026]
for code, config in experiments.items():
    for seed in seeds:
        run_dir = f'{RESULT_ROOT}/outputs/{code}_seed{seed}'
        report_dir = f'{RESULT_ROOT}/reports/{code}_seed{seed}'
        subprocess.run([
            'python', '-m', 'bilinear_lmmd.engine.train', '--config', config,
            '--seed', str(seed), '--output-dir', run_dir,
        ], check=True)
        subprocess.run([
            'python', '-m', 'bilinear_lmmd.engine.evaluate_checkpoint',
            '--checkpoint', f'{run_dir}/best.pt', '--domain', 'source',
            '--split', 'test', '--output-dir', report_dir,
        ], check=True)

In [ ]:
def reports(code):
    return [f'{RESULT_ROOT}/reports/{code}_seed{s}/metrics.json' for s in seeds]

for baseline, candidate in [('M0', 'M1'), ('M0b', 'M1')]:
    output = f'{RESULT_ROOT}/reports/{baseline}_vs_{candidate}_aggregate.json'
    subprocess.run([
        'python', '-m', 'bilinear_lmmd.reporting.aggregate_ablation',
        '--baseline', *reports(baseline), '--candidate', *reports(candidate),
        '--output', output,
    ], check=True)

In [ ]:
import json
from pprint import pprint
for name in ['M0_vs_M1', 'M0b_vs_M1']:
    path = Path(f'{RESULT_ROOT}/reports/{name}_aggregate.json')
    print(f'\n{name}')
    pprint(json.loads(path.read_text())['summary'])